# Telegraph Model: Parameter Exploration

The **telegraph model** describes stochastic gene expression as a two-state
Markov process. A single gene switches between an inactive (**OFF**) and an
active (**ON**) state; transcription occurs only in the ON state:

$$
G_{\text{OFF}}
  \xrightleftharpoons[k_{\text{off}}]{k_{\text{on}}}
G_{\text{ON}}
  \xrightarrow{k_{\text{syn}}}
G_{\text{ON}} + \text{mRNA}
  \xrightarrow{k_{\text{deg}}}
\varnothing
$$

| Parameter | Biological meaning | Typical range |
|---|---|---|
| $k_{\text{on}}$  | Chromatin opening / TF binding rate | 0.001 – 10 min⁻¹ |
| $k_{\text{off}}$ | Chromatin closing / TF unbinding rate | 0.001 – 10 min⁻¹ |
| $k_{\text{syn}}$ | Transcription initiation rate (ON state only) | 1 – 100 min⁻¹ |
| $k_{\text{deg}}$ | mRNA degradation rate (first-order decay) | 0.01 – 1 min⁻¹ |

Below we explore how each parameter shapes the **transient** and
**steady-state** behaviour of the system.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from src.ssa_simulation import compute_sample_moments, simulate_telegraph
from src.ssa_visualization import show_sample_moments

---
## 1. Fast vs. Slow Gene Switching

The **switching regime** is determined by how $k_{\text{on}}$ and
$k_{\text{off}}$ compare to the mRNA degradation rate $k_{\text{deg}}$:

| Regime | Condition | Biological interpretation |
|---|---|---|
| **Slow switching** | $k_{\text{on}}, k_{\text{off}} \ll k_{\text{deg}}$ | The gene stays locked in one state for many mRNA lifetimes. Transcription occurs in long, infrequent **bursts**. This produces a bimodal mRNA distribution — cells are either "empty" or "loaded". |
| **Fast switching** | $k_{\text{on}}, k_{\text{off}} \gg k_{\text{deg}}$ | The gene flips between ON/OFF much faster than mRNA turns over. Each mRNA molecule "sees" a time-averaged gene state. The result is a **unimodal**, near-Poisson distribution. |

### Key observables to watch
- **⟨G⟩** converges to $k_{\text{on}} / (k_{\text{on}} + k_{\text{off}})$ in both cases,
  but the *speed* of convergence differs dramatically.
- **σ_G** is large and persistent in the slow regime (long dwell times),
  but small in the fast regime (rapid averaging).
- **Cov(R, G)** is strong in the slow regime (RNA tracks gene state closely)
  and weak in the fast regime (RNA is decoupled from instantaneous gene state).

### 1a. Slow Switching ($k_{\text{on}} = 0.01,\; k_{\text{off}} = 0.02 \ll k_{\text{deg}} = 0.2$)

Here the gene dwells in each state for ~50–100 time units while mRNA
half-life is only ~3.5 time units. When the gene finally turns ON,
mRNA accumulates rapidly to $k_{\text{syn}}/k_{\text{deg}} = 50$
molecules before the gene switches OFF again. This is the
**transcriptional bursting** regime observed in mammalian cells.

In [ ]:
params_slow = {
    "k_on": 0.01,   # gene activates ~once per 100 time units
    "k_off": 0.02,  # gene inactivates ~once per 50 time units
    "k_syn": 10.0,  # rapid transcription when ON
    "k_deg": 0.2,   # mRNA half-life ≈ 3.5 time units
    "t0": 0.0,
    "g0": 0,        # start in OFF state
    "r0": 0,        # start with zero mRNA
    "n_sim": 4000,
    "n_rep": 300,
}

data_slow = simulate_telegraph(**params_slow)
moments_slow = compute_sample_moments(data_slow)

fig_g, fig_r, fig_c = show_sample_moments(moments_slow)
fig_g.show()
fig_r.show()
fig_c.show()

### 1b. Fast Switching ($k_{\text{on}} = 2.0,\; k_{\text{off}} = 4.0 \gg k_{\text{deg}} = 0.2$)

Now the gene flips ON/OFF every ~0.2–0.5 time units — much faster than
the mRNA lifetime. Individual switching events are invisible at the mRNA
level; the mRNA "sees" a gene that is ON a fraction
$k_{\text{on}}/(k_{\text{on}}+k_{\text{off}}) = 1/3$ of the time.

**Expected differences from the slow regime:**
- ⟨G⟩ converges to the *same* steady state (1/3) but **much faster**.
- σ_G collapses quickly — the ensemble is tightly concentrated.
- Cov(R, G) is very small — knowing the gene state tells you almost
  nothing about the current mRNA count.

In [ ]:
params_fast = {
    "k_on": 2.0,    # gene flips ON every ~0.5 time units
    "k_off": 4.0,   # gene flips OFF every ~0.25 time units
    "k_syn": 10.0,
    "k_deg": 0.2,
    "t0": 0.0,
    "g0": 0,
    "r0": 0,
    "n_sim": 4000,
    "n_rep": 300,
}

data_fast = simulate_telegraph(**params_fast)
moments_fast = compute_sample_moments(data_fast)

fig_g, fig_r, fig_c = show_sample_moments(moments_fast)
fig_g.show()
fig_r.show()
fig_c.show()

---
## 2. High vs. Low Expression Level

The **steady-state mRNA abundance** when the gene is constitutively ON
is set by the ratio $k_{\text{syn}} / k_{\text{deg}}$. This determines
how many mRNA molecules a cell accumulates during an ON burst.

| Scenario | $k_{\text{syn}}/k_{\text{deg}}$ | Biological analogy |
|---|---|---|
| **High expression** | 80 | Abundant housekeeping gene (e.g. *GAPDH*, ribosomal proteins). Many transcripts buffer stochastic fluctuations → coefficient of variation (CV) is low. |
| **Low expression** | 2 | Rare regulatory transcript (e.g. a transcription factor, signalling molecule). Very few molecules → large relative fluctuations (high CV), functionally significant noise. |

We keep the switching rates symmetric ($k_{\text{on}} = k_{\text{off}} = 0.1$)
to isolate the effect of expression level from switching dynamics.

In [ ]:
# Symmetric switching, intermediate regime
base_switching = {
    "k_on": 0.1, "k_off": 0.1,
    "t0": 0.0, "g0": 0, "r0": 0,
    "n_sim": 3000, "n_rep": 400,
}

### 2a. High Expression ($k_{\text{syn}}/k_{\text{deg}} = 80$)

With $k_{\text{syn}} = 40$ and $k_{\text{deg}} = 0.5$, each ON burst
can produce up to 80 mRNA molecules. Even with intermittent gene
silencing, the average mRNA count stays high (~40 at steady state
because the gene is ON half the time). The **large molecule number**
means relative fluctuations ($\sigma_R / \langle R \rangle$) are
modest — the cell has a reliable readout of gene activity.

In [ ]:
params_high = {**base_switching, "k_syn": 40.0, "k_deg": 0.5}

data_high = simulate_telegraph(**params_high)
moments_high = compute_sample_moments(data_high)

fig_g, fig_r, fig_c = show_sample_moments(moments_high)
fig_g.show()
fig_r.show()
fig_c.show()

### 2b. Low Expression ($k_{\text{syn}}/k_{\text{deg}} = 2$)

With $k_{\text{syn}} = 2$ and $k_{\text{deg}} = 1$, the gene can
sustain at most ~2 transcripts during an ON period. Since the gene
is ON only half the time, the steady-state mean is ~1 molecule.

At such low copy numbers:
- **Intrinsic noise dominates**: individual birth/death events cause
  large fractional changes in the mRNA count.
- $\sigma_R$ is comparable to $\langle R \rangle$ — the coefficient
  of variation (CV) approaches 1 or higher.
- This is the regime where **stochastic gene expression** has the
  most biological impact (e.g. cell-fate decisions, phenotypic switching).

In [ ]:
params_low = {**base_switching, "k_syn": 2.0, "k_deg": 1.0}

data_low = simulate_telegraph(**params_low)
moments_low = compute_sample_moments(data_low)

fig_g, fig_r, fig_c = show_sample_moments(moments_low)
fig_g.show()
fig_r.show()
fig_c.show()

---
## 3. Effect of Initial Conditions

The telegraph model is an **ergodic** Markov chain — regardless of
where we start, the system will eventually reach the same stationary
distribution. However, the **transient** behaviour depends strongly
on the initial state.

We test three initial conditions with the same kinetic parameters:

| Scenario | $G_0$ | $R_0$ | Physical interpretation |
|---|---|---|---|
| **Cold start** | 0 (OFF) | 0 | Freshly silenced gene, no pre-existing mRNA. Mimics a newly divided cell or induction from scratch. |
| **Hot start** | 1 (ON) | 50 | Gene already active with a full mRNA load. Mimics steady-state sampling or a highly active locus. |
| **Mismatch** | 0 (OFF) | 50 | Gene is OFF but mRNA is abundant — e.g. right after a burst ends, or an artificial perturbation. mRNA will decay until the gene reactivates. |

**What to observe:** all three should converge to the *same* steady-state
⟨R⟩, but the approach can be from below (cold), from above (mismatch),
or nearly flat (hot).

In [ ]:
ic_base = {
    "k_on": 0.05, "k_off": 0.05,
    "k_syn": 10.0, "k_deg": 0.2,
    "n_sim": 4000, "n_rep": 300, "t0": 0.0,
}

# 3a. Cold start: gene OFF, zero mRNA
data_cold = simulate_telegraph(**{**ic_base, "g0": 0, "r0": 0})
moments_cold = compute_sample_moments(data_cold)

print("── 3a. Cold start (G₀=0, R₀=0) ──")
fig_g, fig_r, fig_c = show_sample_moments(moments_cold)
fig_g.show()
fig_r.show()
fig_c.show()

In [ ]:
# 3b. Hot start: gene ON, high mRNA
data_hot = simulate_telegraph(**{**ic_base, "g0": 1, "r0": 50})
moments_hot = compute_sample_moments(data_hot)

print("── 3b. Hot start (G₀=1, R₀=50) ──")
fig_g, fig_r, fig_c = show_sample_moments(moments_hot)
fig_g.show()
fig_r.show()
fig_c.show()

In [ ]:
# 3c. Mismatch: gene OFF but mRNA abundant
data_mis = simulate_telegraph(**{**ic_base, "g0": 0, "r0": 50})
moments_mis = compute_sample_moments(data_mis)

print("── 3c. Mismatch (G₀=0, R₀=50) ──")
fig_g, fig_r, fig_c = show_sample_moments(moments_mis)
fig_g.show()
fig_r.show()
fig_c.show()

---
## 4. Effect of Numerical Parameters

The Gillespie SSA has two **numerical** parameters that do not change
the underlying biology but affect the **quality** of the simulation:

| Parameter | Role | Trade-off |
|---|---|---|
| `n_sim` (reaction events) | Controls how long (in simulated time) each trajectory runs. More events → longer time horizon → better chance of reaching steady state. | Cost is linear in `n_sim`. If too small, the system may not equilibrate. |
| `n_rep` (trajectories) | Controls the **ensemble size** — how many independent cells we simulate. More replicates → smoother sample moments, tighter confidence on ⟨R⟩ and σ. | Cost is linear in `n_rep`. With too few replicates, the sample mean is noisy and σ-bands are unreliable. |

### What to observe
- **Increasing `n_rep`**: the mean line and σ-bands become smoother.
  With 50 trajectories the curves are jagged; with 1000 they are silky.
- **Increasing `n_sim`**: the time axis extends further, giving the
  system more time to reach steady state. If `n_sim` is too small,
  the plot may cut off during the transient.

In [ ]:
num_base = {
    "k_on": 0.05, "k_off": 0.1,
    "k_syn": 10.0, "k_deg": 0.2,
    "t0": 0.0, "g0": 0, "r0": 0,
}

### 4a. Effect of ensemble size (`n_rep`)

We fix `n_sim = 3000` and vary `n_rep` from 50 (very noisy) to 1000
(highly averaged). As $N_{\text{rep}} \to \infty$, the sample mean
converges to the true ensemble mean by the law of large numbers,
and fluctuations in the estimated $\sigma$ decrease as
$\mathcal{O}(1/\sqrt{N_{\text{rep}}})$.

In [ ]:
for n_rep in [50, 300, 1000]:
    data = simulate_telegraph(**{**num_base, "n_sim": 3000, "n_rep": n_rep})
    moments = compute_sample_moments(data)

    print(f"── n_rep = {n_rep} ──")
    _, fig_r, _ = show_sample_moments(moments)
    fig_r.update_layout(
        title_text=f"<b>b</b>  RNA count — n_rep = {n_rep}"
    )
    fig_r.show()

### 4b. Effect of simulation length (`n_sim`)

We fix `n_rep = 300` and vary `n_sim`. With too few events the
trajectory ends before the system equilibrates — the sample moments
never plateau. With enough events we see a clear steady state.

> **Rule of thumb:** the simulated time should be at least
> $5 \times \max(1/k_{\text{on}},\; 1/k_{\text{off}},\; 1/k_{\text{deg}})$
> for the slowest process to fully relax.

In [ ]:
for n_sim in [500, 2000, 6000]:
    data = simulate_telegraph(**{**num_base, "n_sim": n_sim, "n_rep": 300})
    moments = compute_sample_moments(data)

    print(f"── n_sim = {n_sim} ──")
    _, fig_r, _ = show_sample_moments(moments)
    fig_r.update_layout(
        title_text=f"<b>b</b>  RNA count — n_sim = {n_sim}"
    )
    fig_r.show()